# TextThreat Colab Training Notebook

This notebook trains the thesis models for **TextThreat — AI-Powered Detection of Digital Well-Being Risks with Cybersecurity Analytics**.

It trains:

1. SVM + TF-IDF baseline for Jigsaw multi-label toxicity classification.
2. DistilBERT + LoRA improved classifier for Jigsaw multi-label toxicity classification.
3. DistilBERT + LoRA binary stress classifier for Dreaddit.

It also generates the supporting thesis evidence artifacts: sample events, classification metrics, latency, output-level privacy perturbation, fairness demo/audit, and synthetic co-occurrence results.

**Important:** run this notebook in Colab with a GPU runtime for DistilBERT training.

## 1. Select GPU Runtime

In Colab, go to:

`Runtime -> Change runtime type -> Hardware accelerator -> GPU`

The SVM baseline can run on CPU, but DistilBERT training is much faster with GPU.

In [59]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. DistilBERT training will be slow.')

CUDA available: True
GPU: Tesla T4


## 2. Clone The Repository

This clones the GitHub repository into the Colab machine. If you forked or moved the repo, change `REPO_URL`.

In [60]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/abdulmuksith3/textthreat-poc.git'
BRANCH = 'next-phase'
# Keep this import close to REPO_DIR so the cell works even when run by itself.
from pathlib import Path
REPO_DIR = Path('/content/textthreat-poc')

if not REPO_DIR.exists():
    !git clone --branch $BRANCH --single-branch $REPO_URL $REPO_DIR
else:
    print('Repository already exists:', REPO_DIR)

os.chdir(REPO_DIR)
!git fetch origin $BRANCH
!git checkout $BRANCH
!git pull origin $BRANCH
print('Working directory:', Path.cwd())

Repository already exists: /content/textthreat-poc
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 822 bytes | 137.00 KiB/s, done.
From https://github.com/abdulmuksith3/textthreat-poc
 * branch            next-phase -> FETCH_HEAD
   cd2a12a..ca6671e  next-phase -> origin/next-phase
M	data/exports/sample_textthreat_events.ndjson
D	experiments/results/distilbert_metrics.json
M	experiments/results/latency_metrics.json
M	experiments/results/soar_alerts_log.csv
M	experiments/results/svm_metrics.json
D	models/distilbert_jigsaw/README.md
D	models/distilbert_jigsaw/adapter_config.json
D	models/distilbert_jigsaw/config.json
D	models/distilbert_jigsaw/metrics.json
D	models/distilbert_jigsaw/special_tokens_map.json
D	models/distilbert_jigsaw/tokenizer.json
D	models/distilbert_jigsaw/tokenizer_config.json
D	models/dist

## 3. Install Dependencies

This installs the repository requirements, including Transformers, PEFT/LoRA, Fairlearn, Opacus, MLflow, and Splunk/OpenSearch clients.

If Colab asks to restart the runtime after installation, restart and run the notebook again from this point.

In [61]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt

# Colab currently preinstalls an old torchao build in some runtimes.
# PEFT/LoRA does not require torchao here, and the old version makes LoRA injection fail.
!pip uninstall -y torchao
!python -m pip show torchao || true


## 4. Download Datasets From Shared Google Drive Folder

The CSVs are stored in a public Google Drive folder shared with anyone who has the link. This cell downloads the folder into Colab and copies the dataset files into the repo paths required by the training scripts.

Shared folder:

```text
https://drive.google.com/drive/folders/1I2tIX7Fz4BHa_6mjn8QVBYpBTghBVbbY?usp=sharing
```

Expected local repo paths after the copy step:

```text
data/jigsaw/train.csv
data/dreaddit/dreaddit-train.csv
data/dreaddit/dreaddit-test.csv
```

Raw datasets are used locally in Colab only and should not be committed to Git.


In [62]:
import shutil
import subprocess
import sys
from pathlib import Path

PUBLIC_DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1I2tIX7Fz4BHa_6mjn8QVBYpBTghBVbbY?usp=sharing'
DOWNLOAD_DIR = Path('/content/textthreat_drive_data')

try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gdown'])
    import gdown

if not DOWNLOAD_DIR.exists() or not any(DOWNLOAD_DIR.iterdir()):
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    print('Downloading shared Drive folder...')
    gdown.download_folder(
        url=PUBLIC_DRIVE_FOLDER_URL,
        output=str(DOWNLOAD_DIR),
        quiet=False,
        use_cookies=False,
    )
else:
    print('Using existing downloaded folder:', DOWNLOAD_DIR)

csv_files = sorted(DOWNLOAD_DIR.rglob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'No CSV files found under {DOWNLOAD_DIR}. Check the public Drive link permissions.')

print('CSV files found:')
for csv_path in csv_files:
    print(' -', csv_path)

def choose_csv(label, scorer):
    ranked = sorted(((scorer(path), path) for path in csv_files), reverse=True)
    score, path = ranked[0]
    if score <= 0:
        raise FileNotFoundError(f'Could not identify {label} CSV in downloaded folder. Files: {csv_files}')
    print(f'Selected {label}: {path}')
    return path

def jigsaw_train_score(path):
    name = path.name.lower()
    full = str(path).lower()
    if name == 'train.csv' and 'dreaddit' not in full:
        return 100
    if 'jigsaw' in full and 'train' in name:
        return 90
    if 'toxic' in full and 'train' in name:
        return 80
    return 0

def dreaddit_train_score(path):
    name = path.name.lower()
    full = str(path).lower()
    if name == 'dreaddit-train.csv':
        return 100
    if 'dreaddit' in full and 'train' in name:
        return 90
    return 0

def dreaddit_test_score(path):
    name = path.name.lower()
    full = str(path).lower()
    if name == 'dreaddit-test.csv':
        return 100
    if 'dreaddit' in full and 'test' in name:
        return 90
    return 0

DATASET_COPY_MAP = {
    choose_csv('Jigsaw train.csv', jigsaw_train_score): Path('data/jigsaw/train.csv'),
    choose_csv('Dreaddit train CSV', dreaddit_train_score): Path('data/dreaddit/dreaddit-train.csv'),
    choose_csv('Dreaddit test CSV', dreaddit_test_score): Path('data/dreaddit/dreaddit-test.csv'),
}

for source, target in DATASET_COPY_MAP.items():
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
    print(f'Copied {source} -> {target} ({target.stat().st_size / 1024 / 1024:.2f} MB)')


Using existing downloaded folder: /content/textthreat_drive_data
CSV files found:
 - /content/textthreat_drive_data/dreaddit/dreaddit-test.csv
 - /content/textthreat_drive_data/dreaddit/dreaddit-train.csv
 - /content/textthreat_drive_data/jigsaw/sample_submission.csv
 - /content/textthreat_drive_data/jigsaw/test.csv
 - /content/textthreat_drive_data/jigsaw/test_labels.csv
 - /content/textthreat_drive_data/jigsaw/train.csv
Selected Jigsaw train.csv: /content/textthreat_drive_data/jigsaw/train.csv
Selected Dreaddit train CSV: /content/textthreat_drive_data/dreaddit/dreaddit-train.csv
Selected Dreaddit test CSV: /content/textthreat_drive_data/dreaddit/dreaddit-test.csv
Copied /content/textthreat_drive_data/jigsaw/train.csv -> data/jigsaw/train.csv (66.15 MB)
Copied /content/textthreat_drive_data/dreaddit/dreaddit-train.csv -> data/dreaddit/dreaddit-train.csv (2.60 MB)
Copied /content/textthreat_drive_data/dreaddit/dreaddit-test.csv -> data/dreaddit/dreaddit-test.csv (0.72 MB)


## 5. Confirm Dataset Files

This checks that the expected files exist before training starts.

In [63]:
expected_files = [
    Path('data/jigsaw/train.csv'),
    Path('data/dreaddit/dreaddit-train.csv'),
    Path('data/dreaddit/dreaddit-test.csv'),
]

for path in expected_files:
    print(path, 'exists:', path.exists(), 'size_mb:', round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else 'missing')

data/jigsaw/train.csv exists: True size_mb: 66.15
data/dreaddit/dreaddit-train.csv exists: True size_mb: 2.6
data/dreaddit/dreaddit-test.csv exists: True size_mb: 0.72


## 6. Training Configuration

Use `QUICK_TEST = True` first to verify the end-to-end training workflow quickly.

For final thesis evidence, set:

```python
QUICK_TEST = False
JIGSAW_EPOCHS = 1  # or more if you have time/GPU budget
DREADDIT_EPOCHS = 1
```

LoRA is enabled by default because the thesis describes efficient DistilBERT adaptation.

In [64]:
import shutil
import subprocess
from pathlib import Path

QUICK_TEST = True

# Set these to True/False if you want to run only part of the training pipeline.
RUN_SVM = True
RUN_JIGSAW_DISTILBERT = True
RUN_DREADDIT_DISTILBERT = True

# LoRA trains small adapter layers instead of the full transformer, reducing memory/time.
USE_LORA = True

# One epoch is enough for a thesis PoC run; increase only if you have time/GPU budget.
JIGSAW_EPOCHS = 1
DREADDIT_EPOCHS = 1

# QUICK_TEST uses small subsets so you can verify that everything works.
SVM_SAMPLE_SIZE = 2000 if QUICK_TEST else None
JIGSAW_SAMPLE_SIZE = 1000 if QUICK_TEST else None
DREADDIT_SAMPLE_SIZE = 1000 if QUICK_TEST else None

def run_checked(command):
    print('Running:', command)
    result = subprocess.run(command, shell=True, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print('STDERR:')
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {command}')
    return result

def remove_path(path):
    path = Path(path)
    if path.is_dir():
        shutil.rmtree(path)
        print('Removed old directory:', path)
    elif path.exists():
        path.unlink()
        print('Removed old file:', path)

print('QUICK_TEST:', QUICK_TEST)
print('USE_LORA:', USE_LORA)


QUICK_TEST: True
USE_LORA: True


## 7. Smoke Test Before Heavy Training

This verifies the schema, sample exports, metrics, latency, DP, fairness, co-occurrence, and SOAR-lite demo path before spending GPU time.

In [65]:
!python scripts/smoke_test.py

Wrote classification metrics to /content/textthreat-poc/experiments/results/classification_metrics.json
Wrote latency metrics to /content/textthreat-poc/experiments/results/latency_metrics.json
Wrote DP perturbation results to /content/textthreat-poc/experiments/results/dp_results.json
Wrote fairness results to /content/textthreat-poc/experiments/results/fairness_results.json
Wrote co-occurrence results to /content/textthreat-poc/experiments/results/cooccurrence_results.json
Wrote 3 SOAR-lite alert(s) to /content/textthreat-poc/experiments/results/soar_alerts_log.csv
TextThreat smoke test passed.


## 8. Train SVM + TF-IDF Baseline

This is the thesis baseline model. It uses TF-IDF unigrams+bigrams and a calibrated one-vs-rest LinearSVC.

Output:

- `models/svm_tfidf/`
- `experiments/results/svm_metrics.json`

In [66]:
if RUN_SVM:
    remove_path('models/svm_tfidf')
    remove_path('experiments/results/svm_metrics.json')
    cmd = 'python -m src.textthreat.train_svm --calibration-cv 3'
    if SVM_SAMPLE_SIZE is not None:
        cmd += f' --sample-size {SVM_SAMPLE_SIZE}'
    run_checked(cmd)
else:
    print('Skipping SVM baseline training.')


Removed old directory: models/svm_tfidf
Removed old file: experiments/results/svm_metrics.json
Running: python -m src.textthreat.train_svm --calibration-cv 3 --sample-size 2000
Wrote SVM metrics to /content/textthreat-poc/experiments/results/svm_metrics.json



## 9. Train DistilBERT + LoRA On Jigsaw

This is the improved thesis classifier for six-label Jigsaw harm detection.

LoRA configuration in the repo:

- `r = 8`
- `alpha = 16`
- `dropout = 0.1`
- target modules: `q_lin`, `v_lin`

Output:

- `models/distilbert_jigsaw/`
- `experiments/results/distilbert_metrics.json`

In [67]:
if RUN_JIGSAW_DISTILBERT:
    remove_path('models/distilbert_jigsaw')
    remove_path('experiments/results/distilbert_metrics.json')
    cmd = f'python -m src.textthreat.train_distilbert --task jigsaw --epochs {JIGSAW_EPOCHS}'
    if JIGSAW_SAMPLE_SIZE is not None:
        cmd += f' --sample-size {JIGSAW_SAMPLE_SIZE}'
    if not USE_LORA:
        cmd += ' --no-lora'
    run_checked(cmd)
else:
    print('Skipping Jigsaw DistilBERT training.')


Running: python -m src.textthreat.train_distilbert --task jigsaw --epochs 1 --sample-size 1000
{'loss': '0.6326', 'grad_norm': '1.152', 'learning_rate': '1.02e-05', 'epoch': '0.5'}
{'loss': '0.545', 'grad_norm': '1.128', 'learning_rate': '2e-07', 'epoch': '1'}
{'train_runtime': '10.57', 'train_samples_per_second': '75.69', 'train_steps_per_second': '9.461', 'train_loss': '0.5888', 'epoch': '1'}
Wrote DistilBERT metrics to /content/textthreat-poc/experiments/results/distilbert_metrics.json

STDERR:

Map: 100%|██████████| 1000/1000 [00:00<00:00, 9736.06 examples/s]

Map: 100%|██████████| 800/800 [00:00<00:00, 3526.17 examples/s]

Map: 100%|██████████| 200/200 [00:00<00:00, 3405.01 examples/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2013.16it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-

## 10. Train DistilBERT + LoRA On Dreaddit

This trains the binary stress classifier used by the thesis to support stress signal detection and co-occurrence analytics.

Output:

- `models/distilbert_dreaddit/`
- `experiments/results/dreaddit_metrics.json`

In [68]:
if RUN_DREADDIT_DISTILBERT:
    remove_path('models/distilbert_dreaddit')
    remove_path('experiments/results/dreaddit_metrics.json')
    cmd = f'python -m src.textthreat.train_distilbert --task dreaddit --epochs {DREADDIT_EPOCHS}'
    if DREADDIT_SAMPLE_SIZE is not None:
        cmd += f' --sample-size {DREADDIT_SAMPLE_SIZE}'
    if not USE_LORA:
        cmd += ' --no-lora'
    run_checked(cmd)
else:
    print('Skipping Dreaddit DistilBERT training.')


Running: python -m src.textthreat.train_distilbert --task dreaddit --epochs 1 --sample-size 1000
{'loss': '0.6917', 'grad_norm': '1.594', 'learning_rate': '1.216e-05', 'epoch': '0.4'}
{'loss': '0.6795', 'grad_norm': '1.553', 'learning_rate': '4.16e-06', 'epoch': '0.8'}
{'train_runtime': '10.82', 'train_samples_per_second': '92.38', 'train_steps_per_second': '11.55', 'train_loss': '0.6837', 'epoch': '1'}
Wrote DistilBERT metrics to /content/textthreat-poc/experiments/results/dreaddit_metrics.json

STDERR:

Map: 100%|██████████| 1000/1000 [00:00<00:00, 3269.87 examples/s]

Map: 100%|██████████| 715/715 [00:00<00:00, 3466.61 examples/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3622.46it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.

## 11. Export Full DistilBERT Model Artifacts

The training script saves LoRA adapter-only folders by default. Adapter-only folders contain `adapter_config.json` and `adapter_model.safetensors`, but they do **not** contain the full Hugging Face files required by the artifact checker, especially `config.json` and `model.safetensors`.

This cell merges each LoRA adapter into `distilbert-base-uncased` and saves a complete, self-contained model folder:

```text
models/distilbert_jigsaw/config.json
models/distilbert_jigsaw/model.safetensors
models/distilbert_dreaddit/config.json
models/distilbert_dreaddit/model.safetensors
```

Run this cell even if training has already completed. It is safe to rerun.


In [ ]:
from pathlib import Path
import json
import shutil
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

try:
    from peft import PeftModel
except ImportError as exc:
    raise ImportError("PEFT is required to merge LoRA adapters. Run the dependency installation cell first.") from exc

BASE_MODEL_NAME = "distilbert-base-uncased"

JIGSAW_LABELS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate",
]

DREADDIT_LABELS = [
    "non_stress",
    "stress",
]


def list_top_level_files(model_dir):
    model_dir = Path(model_dir)
    if not model_dir.exists():
        print(f"[MISSING] {model_dir}")
        return
    print(f"\nFiles in {model_dir}:")
    for path in sorted(model_dir.iterdir()):
        if path.is_file():
            print(" -", path.name)


def merge_lora_adapter_to_full_model(
    adapter_dir,
    num_labels,
    label_names,
    problem_type,
    base_model_name=BASE_MODEL_NAME,
):
    """Merge a PEFT/LoRA adapter into DistilBERT and save a full HF model folder.

    This fixes artifact validation failures caused by missing config.json.
    The original adapter files are preserved in an _adapter_backup folder.
    """
    adapter_dir = Path(adapter_dir)
    if not adapter_dir.exists():
        raise FileNotFoundError(f"Model folder does not exist: {adapter_dir}")

    config_path = adapter_dir / "config.json"
    full_weights_path = adapter_dir / "model.safetensors"
    adapter_config_path = adapter_dir / "adapter_config.json"

    if config_path.exists() and full_weights_path.exists():
        print(f"[OK] Full model already exists: {adapter_dir}")
        list_top_level_files(adapter_dir)
        return

    if not adapter_config_path.exists():
        raise FileNotFoundError(
            f"{adapter_dir} is missing adapter_config.json and also missing full model files. "
            "Rerun the DistilBERT training cell."
        )

    print(f"\nMerging LoRA adapter into full model: {adapter_dir}")

    # Keep a backup of adapter files before writing the merged model into the same folder.
    backup_dir = adapter_dir / "_adapter_backup"
    backup_dir.mkdir(parents=True, exist_ok=True)
    for filename in [
        "adapter_config.json",
        "adapter_model.safetensors",
        "adapter_model.bin",
        "README.md",
        "training_args.bin",
    ]:
        src = adapter_dir / filename
        if src.exists():
            dst = backup_dir / filename
            shutil.copy2(src, dst)

    tokenizer = AutoTokenizer.from_pretrained(adapter_dir)

    id2label = {i: label for i, label in enumerate(label_names)}
    label2id = {label: i for i, label in enumerate(label_names)}

    base_model = AutoModelForSequenceClassification.from_pretrained(
        base_model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
        problem_type=problem_type,
        ignore_mismatched_sizes=True,
    )

    peft_model = PeftModel.from_pretrained(base_model, adapter_dir)
    merged_model = peft_model.merge_and_unload()

    # Ensure thesis/inference metadata is stored in config.json.
    merged_model.config.id2label = id2label
    merged_model.config.label2id = label2id
    merged_model.config.problem_type = problem_type

    adapter_dir.mkdir(parents=True, exist_ok=True)
    merged_model.save_pretrained(adapter_dir, safe_serialization=True)
    tokenizer.save_pretrained(adapter_dir)

    # Add a small manifest so it is clear this folder is a merged export.
    manifest = {
        "base_model": base_model_name,
        "export_type": "merged_lora_full_huggingface_model",
        "num_labels": num_labels,
        "problem_type": problem_type,
        "labels": label_names,
        "required_files": ["config.json", "model.safetensors", "tokenizer.json", "tokenizer_config.json"],
    }
    (adapter_dir / "textthreat_model_manifest.json").write_text(json.dumps(manifest, indent=2))

    print(f"[OK] Saved full merged model to {adapter_dir}")
    list_top_level_files(adapter_dir)


if RUN_JIGSAW_DISTILBERT:
    merge_lora_adapter_to_full_model(
        adapter_dir="models/distilbert_jigsaw",
        num_labels=6,
        label_names=JIGSAW_LABELS,
        problem_type="multi_label_classification",
    )
else:
    print("Skipping Jigsaw merge because RUN_JIGSAW_DISTILBERT is False.")

if RUN_DREADDIT_DISTILBERT:
    merge_lora_adapter_to_full_model(
        adapter_dir="models/distilbert_dreaddit",
        num_labels=2,
        label_names=DREADDIT_LABELS,
        problem_type="single_label_classification",
    )
else:
    print("Skipping Dreaddit merge because RUN_DREADDIT_DISTILBERT is False.")


## 12. Generate Supporting Thesis Artifacts

These scripts create the evidence JSON files used by the thesis tables and the SIEM/SOAR demo.

The DP script is intentionally described as **output-level privacy-preserving perturbation**, not full DP-SGD training.

In [69]:
!python -m src.textthreat.export_events --sample
!python -m src.textthreat.evaluate --sample
!python -m src.textthreat.latency --sample
!python -m src.textthreat.dp_output --sample
!python -m src.textthreat.fairness --sample
!python -m src.textthreat.cooccurrence --sample
!python soar_lite/soar_lite.py --demo

Wrote 5 TextThreat events to /content/textthreat-poc/data/exports/sample_textthreat_events.ndjson
Wrote classification metrics to /content/textthreat-poc/experiments/results/classification_metrics.json
Wrote latency metrics to /content/textthreat-poc/experiments/results/latency_metrics.json
Wrote DP perturbation results to /content/textthreat-poc/experiments/results/dp_results.json
Wrote fairness results to /content/textthreat-poc/experiments/results/fairness_results.json
Wrote co-occurrence results to /content/textthreat-poc/experiments/results/cooccurrence_results.json
Wrote 3 SOAR-lite alert(s) to /content/textthreat-poc/experiments/results/soar_alerts_log.csv


## 13. Inspect Result Files

This lists the generated artifacts. These JSON files can be copied into thesis tables or committed when they are demo/sample artifacts.

In [70]:
from pathlib import Path
import json

for path in sorted(Path('experiments/results').glob('*')):
    print(path, round(path.stat().st_size / 1024, 2), 'KB')

print('\nDistilBERT metrics preview:')
metrics_path = Path('experiments/results/distilbert_metrics.json')
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text()) , indent=2)[:2000])
else:
    print('distilbert_metrics.json not found. Check whether training completed.')

experiments/results/classification_metrics.json 15.91 KB
experiments/results/cooccurrence_results.json 0.29 KB
experiments/results/distilbert_metrics.json 0.41 KB
experiments/results/dp_results.json 1.14 KB
experiments/results/dreaddit_metrics.json 0.35 KB
experiments/results/fairness_results.json 3.94 KB
experiments/results/latency_metrics.json 1.13 KB
experiments/results/soar_alerts_log.csv 0.85 KB
experiments/results/svm_metrics.json 17.0 KB

DistilBERT metrics preview:
{
  "demo": false,
  "epoch": 1.0,
  "eval_ece": 0.3663222537438074,
  "eval_loss": 0.5214716196060181,
  "eval_macro_f1": 0.0,
  "eval_macro_pr_auc": 0.03898964737240663,
  "eval_macro_roc_auc": 0.4026603267125237,
  "eval_micro_f1": 0.0,
  "eval_runtime": 1.1005,
  "eval_samples_per_second": 181.736,
  "eval_steps_per_second": 22.717,
  "model_name": "distilbert-base-uncased",
  "task": "jigsaw",
  "use_lora": true
}


## 15. Zip Model And Result Artifacts

This creates a single zip file you can download from Colab. It includes models and result JSONs, but not raw datasets.

## 14. Validate Model Artifact Completeness

This cell checks that trained model directories contain the files needed for inference. After the merge-export step, each DistilBERT folder should include `config.json` and `model.safetensors`. Adapter-only files are not enough for this checker.


In [71]:
# Fail fast if model folders are incomplete before downloading artifacts.
# For LoRA models, adapter_model.safetensors is required alongside adapter_config.json.
!python scripts/check_model_artifacts.py --root .


[FAIL] distilbert_jigsaw
  - Missing config.json in models/distilbert_jigsaw
[OK] svm_tfidf
[FAIL] distilbert_dreaddit
  - Missing config.json in models/distilbert_dreaddit
Model artifact check failed.


In [72]:
# Re-run artifact validation here too, so an incomplete model cannot be zipped by accident.
run_checked('python scripts/check_model_artifacts.py --root .')
run_checked("zip -r textthreat_training_artifacts.zip models experiments/results data/exports/sample_textthreat_events.ndjson schema siem/splunk soar_lite README.md -x '*/__pycache__/*' '*/hf_outputs/*' '*/_adapter_backup/*'")
print('Created textthreat_training_artifacts.zip')


Running: python scripts/check_model_artifacts.py --root .
[FAIL] distilbert_jigsaw
  - Missing config.json in models/distilbert_jigsaw
[OK] svm_tfidf
[FAIL] distilbert_dreaddit
  - Missing config.json in models/distilbert_dreaddit

STDERR:
Model artifact check failed.



RuntimeError: Command failed with exit code 1: python scripts/check_model_artifacts.py --root .

## 16. Download Artifacts

Download the artifact zip to your machine. You can use the trained model folder later in the local or hosted demo.

In [ ]:
from google.colab import files
files.download('textthreat_training_artifacts.zip')